# 🖥️ Local AI Developer Assistant
**Day 10 · AI Application Development Bootcamp**

This lab builds a developer assistant that runs entirely on **your own machine** using Ollama — no cloud API, no internet needed after setup.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Write your reflection in the **📝 Reflection** section at the end

### Sections at a glance

| Part | Topic | Type |
|------|-------|------|
| A | Connect to Ollama | Run & read |
| B | Core generation function | Fill in |
| C | Prompt templates for developer tasks | Fill in |
| D | Multi-turn conversation with `/api/chat` | Fill in |
| E | Task classifier and router | Fill in |
| F | Multi-model benchmark | Fill in |
| G | Optional UI (Streamlit or Gradio) | Open |
| H | Reflection | Write |

### Before you start
Make sure Ollama is running and at least one model is pulled (see handout Section 6):
```bash
ollama serve
```
Suggested models — pick what fits your hardware:

| Model | Min RAM | Notes |
|-------|---------|-------|
| `qwen2.5:1.5b` | 4 GB | Today's default — works on any laptop |
| `llama3.2:1b` | 4 GB | Alternative — equally valid |
| `llama3.2:3b` | 6 GB | Better quality, still fast |
| `qwen3:4b` | 8 GB | Noticeably better if hardware allows |


---
## ⚙️ Setup — Imports


In [1]:
# If needed: !pip install requests pandas

import requests
import time
import pandas as pd
from typing import Dict, Any, List

OLLAMA_URL = "http://localhost:11434"

# ✏️ Change this to whichever model you pulled.
# The benchmark (Part F) will compare this against a second model.
MODEL_NAME = "qwen3:0.6b"

print(f"Using model: {MODEL_NAME}")
print(f"Ollama URL : {OLLAMA_URL}")


Using model: qwen3:0.6b
Ollama URL : http://localhost:11434


---
## Part A — Connect to Ollama

Before writing any prompts, confirm that Ollama is running and the right models are installed.
If this cell fails, run `ollama serve` in a separate terminal and try again.


In [2]:
def check_ollama() -> bool:
    """Return True if Ollama is reachable and list installed models."""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        response.raise_for_status()
        models = response.json().get("models", [])
        print("✅ Ollama is running.")
        print(f"   Installed models ({len(models)}):")
        for m in models:
            print("   -", m.get("name"))
        return True
    except Exception as e:
        print("❌ Could not connect to Ollama.")
        print("   Run `ollama serve` in a separate terminal, then try again.")
        print("   Error:", e)
        return False

check_ollama()


✅ Ollama is running.
   Installed models (3):
   - qwen3:1.7b
   - qwen3:0.6b
   - gemma4:e2b


True

---
## Part B — Core Generation Function

This is the **only place in the whole lab where a network call is made**. Every prompt template, classifier, and benchmark you build in later sections calls this one function.

It uses `/api/generate` — one prompt in, one response out, no memory of previous turns. You will add multi-turn memory in Part D.

**Why `temperature=0.2`?**  
Low temperature means the model picks the most likely next token rather than sampling creatively. For developer tasks — debugging, explanation, test generation — you want **consistent, factual answers**, not creative variation. Try higher values (0.7+) only for open-ended tasks.

**What you should see after filling this in:**  
A short direct answer to the test prompt, printed cleanly.


In [3]:
def ask_ollama(prompt: str, model: str = MODEL_NAME, temperature: float = 0.2) -> str:
    """
    Send a single prompt to a local Ollama model and return the response text.

    Endpoint: POST /api/generate
    Request body: {model, prompt, stream, options: {temperature}}
    Response: response.json()["response"]
    """
    url = f"{OLLAMA_URL}/api/generate"

    # ✏️ YOUR TURN: build the payload dict
    # Keys needed: "model", "prompt", "stream" (set to False), "options" ({"temperature": temperature})
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": ({"temperature": temperature}),
    }
    # ✏️ YOUR TURN: send the POST request, raise on error, return the "response" field
    # Hint: requests.post(url, json=payload, timeout=120)
    #       response.raise_for_status()
    #       return response.json().get("response", "").strip()
    try:
        response = requests.post(url, json=payload, timeout=600)
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

    #raise NotImplementedError("Complete ask_ollama() first.")


# ── Smoke test — uncomment after implementing ─────────────────────────────────
answer = ask_ollama("In one sentence: what is a Python function?")
print("✅", answer)


✅ A Python function is a block of code that performs a specific task, with a name, parameters, and a return value.


---
## Part C — Prompt Templates for Developer Tasks

A prompt template wraps the user's raw input (code, error message, etc.) in a structured instruction that produces consistent, usable output from the model.

**Key principles for developer prompts:**
- Give the model a clear role: `"You are a debugging assistant."`
- Wrap code in a fenced block so the model treats it as code, not prose
- Be explicit about output format — ask for numbered steps, not open-ended paragraphs
- Always ask for a corrected or improved version — makes the output actionable

Implement at least **three** of the four templates below.


In [4]:
def make_code_explainer_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a prompt that explains code to a beginner.

    Include in your prompt:
    - A role instruction  e.g. "You are a helpful programming tutor."
    - The code inside a fenced ```python block
    - Ask for: what the code does, important lines, any bugs or risks,
      and a corrected version if needed
    """
    prompt = """
    You are a helpful programming tutor. 
    You help explain code in simple terms so that beginners can understand it.
    You help point out any potential bugs the code might produce.
    You always ask if the user wants a corrected version of the code. 
    If user responds with 'yes' to wanting a corrected version of the code, you will provide it.
    All output is to be put in numbered steps.
    All code is to be put inside a fenced ```python block.
    The code is as follows: 
    """ + code_snippet

    response = ask_ollama(prompt)
    return response
    
    raise NotImplementedError


def make_debug_prompt(error_message: str, code_snippet: str = "") -> str:
    """
    ✏️ YOUR TURN: write a debugging prompt.

    Include: a role instruction, the error message clearly labelled,
    the code in a fenced block (if provided).
    Ask for: likely cause, problem line, simple fix, safer rewritten version.
    """
    prompt = """
        You are a helpful programming tutor. 
        You help explain why this code produces this error in simple terms so that beginners can understand it.
        You help point out any potential bugs the code might produce.
        You always ask if the user wants a corrected version of the code. 
        If user responds with 'yes' to wanting a corrected version of the code, you will provide it.
        All output is to be put in numbered steps.
        All code is to be put inside a fenced ```python block.
        The code is as follows: 
        """ + code_snippet + """
        The error it created is as follows:
        """ + error_message
    
    response = ask_ollama(prompt)
    return response

    raise NotImplementedError


def make_testcase_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a test-case suggestion prompt.

    For each test case, ask for: input, expected output, why it matters.
    Include edge cases: empty input, None, wrong type, boundary values.
    """
    prompt = """
        You are a helpful programming tutor. 
        You help generate test cases for the code provided.
        You always ask the user for the input, expected output and why it matters. 
        If user provides you the input, expected output and why it matters,
        respond with the generated test cases for the code and all edge cases.
        All output is to be put in numbered steps.
        All code is to be put inside a fenced ```python block.
        The code is as follows: 
        """ + code_snippet
        
    response = ask_ollama(prompt)
    return response

    raise NotImplementedError


def make_improvement_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a code review / improvement prompt.

    Ask for: readability issues, missing error handling, performance notes,
    and a cleaner rewritten version with brief explanations of each change.
    """
    prompt = """
        You are a helpful programming tutor. 
        You help review code and make it so that beginners can understand it.
        You help point out any potential issues that make the code hard to understand 
        such as readability issues, missing error handling and performance notes.
        You always ask if the user wants a cleaner rewritten version of the code. 
        If user responds with 'yes' to wanting a cleaner version of the code, you will provide it.
        All output is to be put in numbered steps.
        All code is to be put inside a fenced ```python block.
        The code is as follows: 
        """ + code_snippet
            
    response = ask_ollama(prompt)
    return response

    raise NotImplementedError


### C2 — Try your templates on a real example

Uncomment each block after implementing the corresponding template. You should see the model produce a structured response for each task type.


In [5]:
sample_code = '''
def divide_numbers(a, b):
    return a / b

print(divide_numbers(10, 0))
'''

sample_error = "ZeroDivisionError: division by zero"

# ── Explanation ───────────────────────────────────────────────────────────────
print("=== 📖 EXPLANATION ===")
print(ask_ollama(make_code_explainer_prompt(sample_code)))

# ── Debug ─────────────────────────────────────────────────────────────────────
print("\n=== 🐛 DEBUG ===")
print(ask_ollama(make_debug_prompt(sample_error, sample_code)))

# ── Test cases ────────────────────────────────────────────────────────────────
print("\n=== 🧪 TEST CASES ===")
print(ask_ollama(make_testcase_prompt(sample_code)))

# ── Improvement ───────────────────────────────────────────────────────────────
print("\n=== ✨ IMPROVEMENT ===")
print(ask_ollama(make_improvement_prompt(sample_code)))


=== 📖 EXPLANATION ===
The function `divide_numbers(a, b)` currently crashes when `b == 0` because it directly returns `a / b`, which is undefined for division by zero. This is a **potential bug** because it could crash the program if called with invalid inputs.

### Why This Happens
- The function is written as a simple return statement: `return a / b`. This means it does not handle the case where division by zero occurs.
- When the function is called with `divide_numbers(10, 0)`, it attempts to perform the division `10 / 0`, which results in a **ZeroDivisionError**.

### How to Fix It
To handle this case, you can add a **try-except** block to catch the `ZeroDivisionError` and return an appropriate message instead of crashing.

```python
def divide_numbers(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: Division by zero"

print(divide_numbers(10, 0))  # Output: Error: Division by zero
```

### Conclusion
The function currently has a bug because

---
## Part D — Multi-turn Conversation with `/api/chat`

So far you have used `/api/generate` — one prompt, one response, no memory of previous turns.

`/api/chat` accepts a `messages` list (same format as OpenAI's API). Each call sends the **full conversation history**, so the model can refer back to earlier messages.

This matters for a developer assistant: a user might say *"explain this function"*, then follow up with *"now add error handling to it"* — and the model needs to know what *"it"* refers to.

**Message format:**
```python
[
    {"role": "system",    "content": "..."},   # optional — sets the model's behaviour
    {"role": "user",      "content": "..."},   # first user message
    {"role": "assistant", "content": "..."},   # model's reply — append this after each turn
    {"role": "user",      "content": "..."},   # next user message
]
```

**What you should see:** Turn 1 explains the function. Turn 2 produces a rewritten version with error handling — and references what was discussed in Turn 1, without you repeating the code.


In [6]:
def chat_with_ollama(
    messages: List[Dict[str, str]],
    model: str = MODEL_NAME,
    temperature: float = 0.2
) -> str:
    """
    Multi-turn chat using /api/chat.
    Returns the model's latest reply as a string.

    Endpoint: POST /api/chat
    Request body: {model, messages, stream: False, options: {temperature}}
    Response: response.json()["message"]["content"]
    """
    # ✏️ YOUR TURN
    # Hint: almost identical to ask_ollama, but:
    #   - use /api/chat instead of /api/generate
    #   - the body key is "messages" (the list), not "prompt"
    #   - the response field is response.json()["message"]["content"]
    url = f"{OLLAMA_URL}/api/chat"
    payload = {
            "model": model,
            "messages": messages,
            "stream": False,
            "options": ({"temperature": temperature}),
        }
    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["message"]["content"]
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

    raise NotImplementedError


# ── Two-turn demo — uncomment after implementing chat_with_ollama() ────────────
conversation = [
    {"role": "system", "content": "You are a helpful Python tutor."},
    {"role": "user",   "content": f"Explain this function:\n```python\n{sample_code}\n```"}
]
#
reply1 = chat_with_ollama(conversation)
print("Turn 1 — Explanation:")
print(reply1)
#
# # ✏️ YOUR TURN: append reply1 as role="assistant", then add the follow-up question
conversation.append({"role": "assistant", "content": reply1})
conversation.append({"role": "user", "content": "Now rewrite it with proper error handling."})
#
reply2 = chat_with_ollama(conversation)
print("\nTurn 2 — Rewrite:")
print(reply2)


Turn 1 — Explanation:
The function `divide_numbers(a, b)` is a simple division function that takes two parameters, `a` and `b`, and returns the result of dividing `a` by `b`. Here's a breakdown:

1. **Function Signature**: The function takes two arguments, `a` and `b`, which are numbers (integers or floats).
2. **Return Value**: It returns the result of the division: `a / b`.
3. **Error Handling**: When called with `a = 10` and `b = 0`, it raises a `ZeroDivisionError`, which is a Python exception. This is because dividing by zero is undefined in mathematics and causes an error.

### Example Execution:
```python
print(divide_numbers(10, 0))  # This will raise ZeroDivisionError: division by zero
```

### Explanation:
- The function is designed to perform basic division.
- When called with `a = 10` and `b = 0`, it throws an error, indicating that the division is undefined.
- The function is straightforward and handles division, but it does not handle division by zero gracefully.

Turn 2 —

---
## Part E — Task Classifier and Router

A real developer assistant should not ask the user to pick a task from a dropdown. It reads the request, decides what kind of task it is, selects the right prompt template automatically, and responds.

This section builds that in two steps:
1. **`classify_task()`** — reads the user's request and returns a task type string
2. **`developer_assistant()`** — uses the task type to select a prompt template and return the answer

The classifier is keyword-based — `if`/`elif` on lowercased text. This is intentional: you don't need machine learning for well-defined categories with reliable keywords. The important design point is **separating classification from generation** — the classifier never calls the model, and the generator doesn't need to know about classification.

**What you should see:** each test request is routed to a different task type and produces the right kind of response.


In [7]:
def classify_task(user_request: str) -> str:
    """
    Classify a free-text developer request into one of four task types.
    Returns one of: "debug", "test", "improve", "explain"

    ✏️ YOUR TURN:
    - Lowercase the request
    - Check for debug keywords:   error, bug, fix, debug, exception, traceback, fails
    - Check for test keywords:    test, case, assert, pytest, unittest, coverage
    - Check for improve keywords: improve, refactor, clean, optimise, optimize, rewrite, review
    - Default to "explain" if none match

    Hint: use any(k in text for k in ["error", "bug", ...])
    """
    lower_user_request = user_request.lower()
    check_debug = any(k in lower_user_request for k in ["error", "bug", "fix", "debug", "exception", "traceback", "fails"])
    if check_debug:
        return "debug"
    check_test = any(k in lower_user_request for k in ["test", "case", "assert", "pytest", "unittest", "coverage"])
    if check_test:
        return "test"
    check_improve = any(k in lower_user_request for k in ["improve", "refactor", "clean", "optimise", "optimize", "rewrite", "review"])
    if check_improve:
        return "improve"
    return "explain"
    
    # YOUR CODE HERE
    raise NotImplementedError


def developer_assistant(
    user_request: str,
    code_snippet: str = "",
    error_message: str = ""
) -> Dict[str, str]:
    """
    Full pipeline: classify request → select prompt template → generate answer.
    Returns {"task": task_type, "answer": model_response}
    so you can see both what was classified and what the model said.

    ✏️ YOUR TURN:
    - Call classify_task(user_request)
    - Select the right make_*_prompt() based on the task type
      (for "debug", pass error_message and code_snippet)
    - Call ask_ollama() with the prompt
    - Return {"task": task, "answer": answer}
    """
    task = classify_task(user_request)
    match task:
        case "debug":
            output = {"task": task, "answer": ask_ollama(make_debug_prompt(error_message, code_snippet))}
            return output
        case "test":
            output = {"task": task, "answer": ask_ollama(make_testcase_prompt(code_snippet))}
            return output
        case "improve":
            output = {"task": task, "answer": ask_ollama(make_improvement_prompt(code_snippet))}
            return output
        case _:
            output = {"task": task, "answer": ask_ollama(make_code_explainer_prompt(code_snippet))}
            return output


    # YOUR CODE HERE
    raise NotImplementedError


# ── Test the router on four requests — uncomment after implementing ────────────
requests_to_test = [
    ("Please debug this",       sample_code, sample_error),
    ("Suggest test cases",      sample_code, ""),
    ("Refactor this function",  sample_code, ""),
    ("What does this do?",      sample_code, ""),
]
#
for req, code_s, err in requests_to_test:
    result = developer_assistant(req, code_s, err)
    print(f"Request  : {req}")
    print(f"Detected : {result['task']}")
    print(f"Answer   : {result['answer'][:250]}...")
    print()


Request  : Please debug this
Detected : debug
Answer   : The error occurs because dividing by zero is undefined in Python. To fix this, we need to ensure that the function handles division by zero properly. Here's a corrected version of the code:

```python
def divide(a, b):
    if b == 0:
        raise Ze...

Request  : Suggest test cases
Detected : test
Answer   : ```python
# Test cases for the divide_numbers function
test_cases = [
    ("10", 0),  # Edge case: division by zero
    ("5", 2),   # Correct division
    ("0", 5),   # Edge case: division by zero
    ("100", 25),  # Correct division
    ("1000", 10)...

Request  : Refactor this function
Detected : improve
Answer   : The function `divide_numbers(a, b)` is designed to perform division and handle division by zero gracefully. Here's a step-by-step explanation:

1. **Error Handling:**  
   The function first checks if `b` is zero. If it is, it raises a `ZeroDivisionE...

Request  : What does this do?
Detected : explain
Answer  

---
## Part F — Multi-Model Benchmark

One of the key questions in local AI is: **how much quality do you give up by using a smaller model, and is the speed gain worth it?**

This section makes that concrete: run the same three prompts through two different model sizes and compare results side by side.

**What to think about while reading the results:**
- Is the quality difference noticeable on the simple task? What about the complex one?
- At which task does the smaller model fall apart first?
- If you were building a real coding assistant, which model would you choose — and why?

**What you should see:** a DataFrame with elapsed times and response lengths per model,
plus two full answers side-by-side for the complex task.


In [8]:
# Set your two models. Pick one size up or down from your primary model.
# The models must both be already pulled — `ollama list` to check.
#
# Example pairs:
#   ("qwen2.5:1.5b",  "qwen2.5:0.5b")   — compare within Qwen 2.5
#   ("llama3.2:3b",   "llama3.2:1b")    — compare within Llama 3.2
#   ("qwen3:4b",      "qwen2.5:1.5b")   — compare across families
MODEL_A = "qwen3:0.6b"           # your primary model
MODEL_B = "qwen3:1.7b"     # ✏️ change to a second model you have pulled


def benchmark_prompt(prompt: str, model: str) -> Dict[str, Any]:
    """
    Run one prompt through one model and record timing and output length.

    ✏️ YOUR TURN:
    - Record start time with time.perf_counter()
    - Call ask_ollama(prompt, model=model)
    - Record end time and compute elapsed
    - Return: {"model", "prompt_chars", "response_chars", "elapsed_seconds", "answer"}
    """
    # YOUR CODE HERE
    start_time = time.perf_counter()
    response = ask_ollama(prompt, model)
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    answer = {
        "model": model,
        "prompt_chars": len(prompt),
        "response_chars": len(response),
        "elapsed_seconds": execution_time,
        "answer": response
    }
    print(answer)
    return answer

    raise NotImplementedError


benchmark_prompts = [
    ("Simple",
     "Explain recursion in Python in one short paragraph."),

    ("Debug",
     f"Find the bug in this code and explain why it fails:\n```python\n{sample_code}\n```"),

    ("Complex",
     "Write a Python function that reads a CSV file, filters rows where a given column "
     "exceeds a threshold, and returns the result as a list of dicts. "
     "Include type hints, a docstring, and error handling for missing files."),
]

# ✏️ YOUR TURN: run all prompts through both models and build a comparison DataFrame.
#
results = []
for label, prompt in benchmark_prompts:
    for model in [MODEL_A, MODEL_B]:
        row = benchmark_prompt(prompt, model)
        row["task"] = label
        results.append(row)
#
df = pd.DataFrame(results)
df[["task", "model", "elapsed_seconds", "response_chars"]]


{'model': 'qwen3:0.6b', 'prompt_chars': 51, 'response_chars': 613, 'elapsed_seconds': 3.5134451999911107, 'answer': 'Recursion in Python is a technique where a function calls itself to solve a smaller subproblem. It works by breaking down a complex task into smaller, similar tasks, with each call reducing the problem size until a base case is reached. For example, a function that counts down from a number to 1 would call itself with a lower number, like `count(5)` calling `count(4)`, and so on, until it reaches the base case where the function returns. The base case is when the input is zero, and the function stops calling itself, returning the result. This allows the function to process multiple subproblems efficiently.'}
{'model': 'qwen3:1.7b', 'prompt_chars': 51, 'response_chars': 774, 'elapsed_seconds': 5.894735299996682, 'answer': "Recursion is a programming technique where a function calls itself to solve a smaller instance of the same problem. It involves breaking down the probl

,task,model,elapsed_seconds,response_chars
0,Simple,qwen3:0.6b,3.513445,613
1,Simple,qwen3:1.7b,5.894735,774
2,Debug,qwen3:0.6b,3.768513,923
3,Debug,qwen3:1.7b,5.809871,1508
4,Complex,qwen3:0.6b,10.235931,1285
5,Complex,qwen3:1.7b,21.275581,2866


### F2 — Quality comparison

Read both model answers for the **Complex** task side by side and rate each (1 = poor, 5 = excellent).

| Task | Model A answer | Model B answer | Notes |
|------|---------------|----------------|-------|
| Simple | 3/5 | 4/5 | Both answer are similar but B gives a warning about infinite loops. |
| Debug | 3/5 | 4/5 | Both warns about divinding by 0 but Model B explains it in steps. |
| Complex | 3/5 | 5/5 | Both returns a python function but Model B add error handling and better explanation. |


In [9]:
# ── Print full answers for the Complex task side by side ─────────────────────
# Uncomment after the benchmark cell runs successfully.

complex_prompt = benchmark_prompts[2][1]
#
for model in [MODEL_A, MODEL_B]:
    print(f"{'='*60}")
    print(f"  {model}")
    print(f"{'='*60}")
    print(ask_ollama(complex_prompt, model=model))
    print()


  qwen3:0.6b
```python
import csv

def read_csv_file(column_name, threshold):
    """
    Reads a CSV file, filters rows where a given column exceeds a threshold, and returns the result as a list of dicts.

    Parameters:
    column_name (str): The name of the column to filter.
    threshold (int/float): The value to exceed.

    Returns:
    list of dicts: A list of rows where the given column exceeds the threshold.

    Raises:
    FileNotFoundError: If the file is not found.
    """
    try:
        with open(file_path, 'r') as file:
            reader = csv.reader(file)
            result = []
            for row in reader:
                if row[column_name] > threshold:
                    result.append(row)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")

# Example usage:
# result = read_csv_file('column_name', 10)
# print(result)
```

**Explanation:**

- **Type Hints:** The function parameters are declared with type hints (`str` fo

---
## Part G — Optional: UI with Streamlit or Gradio

If time allows, wrap `developer_assistant()` in a simple web UI. The goal is to confirm the same local model works through a browser interface — not to build a polished product.

Save either snippet as a `.py` file and run it from the terminal.

### Option A — Streamlit

```python
# dev_assistant_app.py
# Run with: streamlit run dev_assistant_app.py

import streamlit as st
# import your functions from this notebook or copy them here

st.title("🖥️ Local AI Developer Assistant")
st.caption("Running on Ollama — no internet needed")

code_input    = st.text_area("Paste your code here", height=200)
error_input   = st.text_input("Error message (optional)")
request_input = st.text_input("What do you want?  e.g. explain this, debug this, suggest tests")

if st.button("Run assistant") and request_input:
    with st.spinner("Thinking locally..."):
        result = developer_assistant(request_input, code_input, error_input)
    st.markdown(f"**Classified as:** `{result['task']}`")
    st.markdown(result["answer"])
```

### Option B — Gradio

```python
# dev_assistant_gradio.py
# Run with: python dev_assistant_gradio.py

import gradio as gr
# import your functions from this notebook or copy them here

def run(request, code, error):
    result = developer_assistant(request, code, error)
    return f"Classified as: {result['task']}\n\n{result['answer']}"

gr.Interface(
    fn=run,
    inputs=[
        gr.Textbox(label="What do you want?"),
        gr.Textbox(label="Code (optional)", lines=8),
        gr.Textbox(label="Error message (optional)"),
    ],
    outputs=gr.Textbox(label="Answer", lines=15),
    title="Local AI Developer Assistant"
).launch()
```


---
## 📝 Reflection

Write **150–250 words** addressing the questions below. Connect your answers to what you actually observed in Parts C – F — not just what the handout says.

1. **Model choice and hardware:** which model(s) did you use, and why that size given your machine?
2. **Benchmark results:** was the speed difference between your two models significant? At which task did quality diverge most?
3. **Real-world relevance:** from the handout's examples — Samsung data breach, air-gapped systems, cost at scale — which is most relevant to a project you could imagine building, and why?
4. **When to use cloud:** when would you still choose Groq or OpenAI over a local model for a developer assistant?


*Your reflection (150–250 words):*


